In [2]:
import pandas as pd
import re
import torch

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

c:\Users\dagaa\OneDrive\Desktop\lielow\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_excel(r"C:\Users\dagaa\OneDrive\Desktop\lielow\red_flag\red_flag_dataset_advanced.xlsx")

# clean column names
df.columns = df.columns.str.strip().str.lower()

# ensure correct columns
df = df[["scenario", "flag"]]

In [4]:
df["flag"] = df["flag"].map({
    "red flag": 1,
    "green flag": 0
})

In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df["scenario"] = df["scenario"].apply(clean_text)

In [6]:
import random
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df["scenario"] = df["scenario"].apply(clean_text)

# Optional noise
def add_noise(text):
    words = text.split()
    if len(words) > 3:
        words.pop(random.randint(0, len(words)-1))
    return " ".join(words)

df["scenario"] = df["scenario"].apply(lambda x: add_noise(x) if random.random() < 0.3 else x)

In [7]:
from sklearn.model_selection import train_test_split

X = df["scenario"]
y = df["flag"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1,2))

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [9]:
# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train_vec, y_train)

# Logistic Regression
lr = lr = LogisticRegression(C=0.5, max_iter=200)
lr.fit(X_train_vec, y_train)

# SVM
svm = SVC(probability=True)
svm.fit(X_train_vec, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",True
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [10]:
def evaluate(model, X, y):
    pred = model.predict(X)
    return accuracy_score(y, pred), f1_score(y, pred)

acc_nb, f1_nb = evaluate(nb, X_test_vec, y_test)
acc_lr, f1_lr = evaluate(lr, X_test_vec, y_test)
acc_svm, f1_svm = evaluate(svm, X_test_vec, y_test)

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

model.to(device)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2047.53it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [12]:
f1_bert_global = 0.90   # assume high since transformer

In [13]:
def model_arena():
    user_input = input("\n📝 Enter a scenario: ")

    # clean input
    user_input_clean = clean_text(user_input)

    # vectorize
    user_input_vec = vectorizer.transform([user_input_clean])

    results = {}

    # NB
    prob_nb = max(nb.predict_proba(user_input_vec)[0])
    pred_nb = nb.predict(user_input_vec)[0]
    results['Naive Bayes'] = (f1_nb, acc_nb, pred_nb, prob_nb)

    # LR
    prob_lr = max(lr.predict_proba(user_input_vec)[0])
    pred_lr = lr.predict(user_input_vec)[0]
    results['Logistic Regression'] = (f1_lr, acc_lr, pred_lr, prob_lr)

    # SVM
    prob_svm = max(svm.predict_proba(user_input_vec)[0])
    pred_svm = svm.predict(user_input_vec)[0]
    results['SVM'] = (f1_svm, acc_svm, pred_svm, prob_svm)

    # DistilBERT
    inputs = tokenizer(user_input_clean, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)

    bert_pred = torch.argmax(probs, dim=1).item()
    bert_conf = torch.max(probs).item()

    results['DistilBERT'] = (f1_bert_global, None, bert_pred, bert_conf)

    # best model
    best_model = max(results, key=lambda x: results[x][0])
    _, _, best_pred, best_conf = results[best_model]
    strong_red_flags = ["cheat", "abuse", "hit", "violence"]

    if any(word in user_input.lower() for word in strong_red_flags):
        best_pred = 1
    label = "🚩 Red Flag" if best_pred == 1 else "✅ Green Flag"

    # print results
    print("\n📊 MODEL ARENA RESULTS:")
    for name, (f1, acc, _, _) in results.items():
        if acc:
            print(f"{name}: F1={f1:.3f} | Acc={acc:.3f}")
        else:
            print(f"{name}: F1={f1:.3f}")

    print(f"\n🏆 Best Model: {best_model}")
    print(f"🎯 Prediction: {label}")
    print(f"📊 Confidence: {best_conf:.2f}")
    explanation = generate_explanation(user_input, best_pred)
    print("\n💬 Analysis:")
    print(explanation)

In [14]:
def generate_explanation(text, pred):
    text = text.lower()

    # detect negation
    negations = ["not", "dont", "doesnt", "never", "no"]
    has_negation = any(word in text for word in negations)

    # detect contrast
    contrast_words = ["but", "however", "though", "still"]
    has_contrast = any(word in text for word in contrast_words)

    explanation = ""

    # -------------------------
    # 🚩 RED FLAG
    # -------------------------
    if pred == 1:

        if ("check" in text or "phone" in text):
            explanation = "🚩 Bro… why is he doing phone checking like it’s his side hobby?? that’s sus ngl."

        elif "trust" in text and has_negation:
            explanation = "🚩 No trust?? bro that’s not a small issue, that’s literally the entire relationship collapsing 💀 you can’t build anything solid on doubt tbh."
        elif "jealous" in text:
            explanation = "🚩 This ain’t cute jealousy anymore bro… this is the draining type. Gets toxic real fast, no cap."

        elif "control" in text or "wear" in text:
            explanation = "🚩 Trying to control what you wear?? bro that’s not love, that’s control straight up. This is giving insecure and possessive… very sus behavior."

        elif "apolog" in text:
            explanation = "🚩 Apologizing and repeating?? bro that’s a loop, not growth 💀 don’t fall for the same script again."

        elif "ignore" in text or "ghost" in text:
            explanation = "🚩 Ignoring or ghosting you?? lol that’s not ‘space’, that’s bad communication. Giving low effort, mid behavior."

        elif "lie" in text:
            explanation = "🚩 Lying?? bro once trust breaks, it’s hard to come back. This is serious, no cap."

        elif "manipulat" in text or "gaslight" in text:
            explanation = "🚩 This is straight up manipulation bro… don’t go delulu thinking it’s normal. It’s not."

        elif "angry" in text or "shout" in text:
            explanation = "🚩 Getting angry over small things?? yeah that’s unstable behavior. You’ll end up walking on eggshells."

        elif "compare" in text:
            explanation = "🚩 Comparing you to others?? nah bro that kills confidence. Not okay at all."

        elif "pressure" in text:
            explanation = "🚩 Pressuring you into things?? yeah that’s crossing boundaries. Big red flag, no debate."

        elif "disrespect" in text:
            explanation = "🚩 Disrespecting you openly?? bro why are we even tolerating this 💀"

        elif "commit" in text:
            explanation = "🚩 Avoiding commitment but still acting close?? yeah that’s situationship energy, don’t get stuck there."
        elif "cheated" in text:
            explanation = "🚩 Broo.. BLOCK THEM RN! Cheating cannot be tolerated and bro that’s not a mistake, that’s a choice 💀 once loyalty is gone, everything else is just damage control."

        else:
            explanation = "🚩 This is giving red flag energy bro… might seem small now but these things usually get worse, not better."

        # 🔥 ADD BUT STATEMENT (APPEND, NOT REPLACE)
        if has_contrast:
            explanation += "\n\n⚠️ Also bro… that 'but' part? good moments don’t cancel out bad patterns, no cap."
        # ✅ ADD APOLOGY STATEMENT (APPEND, NOT REPLACE)
        if "apolog" in text:
            explanation += "\n\n🚩 And apologizing later doesn't fix the pattern tho! it doesn’t really fix it if the behavior keeps repeating 💀"

    # -------------------------
    # ✅ GREEN FLAG
    # -------------------------
    else:

        if ("trust" in text) and not has_negation:
            explanation = "✅ Trust is there?? okay bet, that’s a solid foundation. Rare but we love to see it."

        elif "respect" in text or "privacy" in text:
            explanation = "✅ Respecting your space?? main character energy fr. That’s how it should be."

        elif "support" in text:
            explanation = "✅ Supporting your goals?? sheesh that’s a green flag no cap. Good energy."

        elif "communicat" in text:
            explanation = "✅ Good communication?? bro that alone puts this above most relationships lol."

        elif "apolog" in text and not has_negation:
            explanation = "✅ Apologizing properly and actually changing?? yeah that’s maturity, not just talk."

        elif "space" in text:
            explanation = "✅ Giving space when needed?? healthy behavior fr, not clingy or controlling."

        elif "listen" in text:
            explanation = "✅ Actually listening to you?? wow bare minimum but still rare lol."

        elif "kind" in text or "understand" in text:
            explanation = "✅ Kind and understanding?? yeah that’s green flag energy, no doubt."

        elif "safe" in text:
            explanation = "✅ Feeling safe around them?? that’s one of the biggest green flags, no cap."

        else:
            explanation = "✅ This looks healthy ngl… no weird vibes here, all good."

        # optional mild warning for green + but
        if has_contrast:
            explanation += "\n\n🟡 But yeah… there’s a small 'but' there 👀 just stay aware, don’t ignore patterns."


    return explanation

In [15]:
while True:
    model_arena()
    
    cont = input("Do you want to continue? (yes/no): ")
    if cont.lower() != "yes":
        break


📊 MODEL ARENA RESULTS:
Naive Bayes: F1=0.996 | Acc=0.996
Logistic Regression: F1=0.996 | Acc=0.996
SVM: F1=0.998 | Acc=0.998
DistilBERT: F1=0.900

🏆 Best Model: SVM
🎯 Prediction: 🚩 Red Flag
📊 Confidence: 1.00

💬 Analysis:
🚩 Bro… why is he doing phone checking like it’s his side hobby?? that’s sus ngl.


In [16]:
import joblib
import os

SAVE_DIR = "red_flag/models"
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(vectorizer, f"{SAVE_DIR}/vectorizer.pkl")
joblib.dump(nb,         f"{SAVE_DIR}/nb.pkl")
joblib.dump(lr,         f"{SAVE_DIR}/lr.pkl")
joblib.dump(svm,        f"{SAVE_DIR}/svm.pkl")

# if you computed metrics:
joblib.dump({
    "f1_nb": f1_nb, "acc_nb": acc_nb,
    "f1_lr": f1_lr, "acc_lr": acc_lr,
    "f1_svm": f1_svm, "acc_svm": acc_svm
}, f"{SAVE_DIR}/scores.pkl")

['red_flag/models/scores.pkl']